In [ ]:
#Importing dependencies 

import numpy as np
import pandas as pd
import os
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# 🧾 Load and preprocess image dataset
def load_data(df, base_dir, img_size=(224, 224)):
    images = []
    labels = []

    for _, row in df.iterrows():
        img_path = os.path.join(base_dir, row["main_frame"])
        try:
            img = load_img(img_path, color_mode="grayscale", target_size=img_size)
            image = img_to_array(img) / 255.0
            images.append(image)

            # ⬇️ Multi-output labels (5 regression targets)
            labels.append([
                row["UPDRS_score"]

            ])
        except Exception as e:
            print(f"Error loading image: {img_path}\n{e}")

    X = np.array(images)
    y = np.array(labels, dtype=np.float32)
    return X, y

# 📁 Setup your data path and dataframe

#Data augmentation 
#data_augmentation = keras.Sequential([
   # layers.RandomFlip("horizontal"),
   # layers.RandomRotation(0.1),
    #layers.RandomZoom(0.1),
   # layers.RandomTranslation(0.1, 0.1),
   # layers.RandomContrast(0.1),


base_dir = "/users/imbahndu/Desktop/Columbia DBM/PFED5/"  # 🛠️ Replace with your actual image directory
df = pd.read_csv("/users/imbahndu/Desktop/Columbia DBM/PFED5/df.csv")  # 🛠️ Your CSV must include 'main_frame' and score columns

# 🧪 Load into memory
X, y = load_data(df, base_dir)

# 🧠 CNN Model
model = keras.Sequential()

model.add(keras.layers.InputLayer(input_shape=(224, 224, 1)))

model.add(keras.layers.Conv2D(32, 3, padding='same'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.ReLU())
model.add(keras.layers.MaxPooling2D((2, 2)))

model.add(keras.layers.Conv2D(64, 3, padding='same'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.ReLU())
model.add(keras.layers.MaxPooling2D((2, 2)))

model.add(keras.layers.Conv2D(128, 3, padding='same'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.ReLU())

model.add(keras.layers.GlobalAveragePooling2D())
model.add(keras.layers.Dropout(0.4))

# 🎯 Multi-output regression (5 continuous targets)
model.add(keras.layers.Dense(5, activation="linear"))

# 📦 Compile model
model.compile(optimizer="adam", loss="mse", metrics=["mse"])
model.summary()

# 🏋️ Train
model.fit(X, y, batch_size=32, epochs=10, validation_split=0.2)


/users/imbahndu/.local/lib/python3.9/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 224, 224, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_12 (ReLU)                 │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_13 (ReLU)                 │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_14 (ReLU)                 │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,213 (368.02 KB)

 Trainable params: 93,765 (366.27 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 167s 2s/step - loss: 0.2586 - mse: 0.2586 - val_loss: 0.1183 - val_mse: 0.1183
Epoch 2/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - loss: 0.0962 - mse: 0.0962 - val_loss: 0.1192 - val_mse: 0.1192
Epoch 3/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - loss: 0.0818 - mse: 0.0818 - val_loss: 0.1280 - val_mse: 0.1280
Epoch 4/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - loss: 0.0769 - mse: 0.0769 - val_loss: 0.1363 - val_mse: 0.1363
Epoch 5/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - loss: 0.0757 - mse: 0.0757 - val_loss: 0.1305 - val_mse: 0.1305
Epoch 6/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - loss: 0.0774 - mse: 0.0774 - val_loss: 0.1422 - val_mse: 0.1422
Epoch 7/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - loss: 0.0745 - mse: 0.0745 - val_loss: 0.1230 - val_mse: 0.1230
Epoch 8/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - loss: 0.0705 - mse: 0.0705 - val_loss: 0.1230 - val_mse: 0.1230
Epoch 9/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - loss: 0.067